# Embeddings — Hands-On

**LLM Engineering · Domain 1 · Roadmap Week 09**

Companion to `02 Literature Notes/LLM Engineering/Embeddings` and the deck
`Lesson_03_Embeddings.pptx`. Runs fully offline (deterministic fallback embedder);
set `OPENAI_API_KEY` for true semantic vectors.

**Sources:** OpenAI text-embedding-3 docs; Sentence-BERT (arXiv:1908.10084);
MTEB (arXiv:2210.07316); ColBERT.

## 0. Setup — an embedder that works with or without an API key

In [ ]:
%pip install -q numpy openai
import os, re, numpy as np
np.set_printoptions(precision=3, suppress=True)

def embed(texts, model="text-embedding-3-small", dims=None):
    if os.getenv("OPENAI_API_KEY"):
        from openai import OpenAI
        kw = {"dimensions": dims} if dims else {}
        r = OpenAI().embeddings.create(model=model, input=texts, **kw)
        v = np.array([d.embedding for d in r.data])
    else:
        dim = dims or 256; v = []
        for t in texts:                       # offline lexical fallback
            x = np.zeros(dim)
            for w in re.findall(r"[a-z0-9]+", t.lower()):
                x[hash(w) % dim] += 1.0
            v.append(x)
        v = np.array(v)
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)   # normalize

print("using", "OpenAI" if os.getenv("OPENAI_API_KEY") else "offline fallback")

## 1. Similarity is geometry
After normalization, cosine similarity is just a dot product.

In [ ]:
def cosine(a, b): return float(a @ b)

V = embed(["car", "automobile", "banana", "reset my password", "recover my login"])
labels = ["car","automobile","banana","reset pwd","recover login"]
import itertools
for i, j in itertools.combinations(range(len(V)), 2):
    print(f"{labels[i]:12s} ~ {labels[j]:12s} : {cosine(V[i], V[j]):.3f}")

> With an API key, `car~automobile` and `reset pwd~recover login` score high while
`car~banana` is low — semantic proximity. The offline fallback only sees lexical overlap.

## 2. Prove: normalize -> dot product == cosine, and Euclidean is monotonic

In [ ]:
raw = np.random.RandomState(0).randn(2, 8) * np.array([[1.0],[5.0]])  # different magnitudes
def cos_raw(a,b): return (a@b)/(np.linalg.norm(a)*np.linalg.norm(b))
un = raw / np.linalg.norm(raw, axis=1, keepdims=True)
print("cosine(raw)            :", round(cos_raw(raw[0], raw[1]), 4))
print("dot(normalized)        :", round(float(un[0] @ un[1]), 4))
print("2 - 2*dot  vs  ||a-b||²:",
      round(2 - 2*float(un[0]@un[1]), 4), round(float(np.sum((un[0]-un[1])**2)), 4))

## 3. Minimal semantic search (the RAG retriever nucleus)

In [ ]:
class SemanticIndex:
    def __init__(self, docs):
        self.docs = docs; self.vectors = embed(docs)
    def search(self, query, k=3):
        q = embed([query])[0]
        scores = self.vectors @ q
        top = np.argsort(-scores)[:k]
        return [(self.docs[i], round(float(scores[i]),3)) for i in top]

corpus = [
  "To reset your password, click 'Forgot password' on the login page.",
  "Our return policy allows refunds within 30 days of purchase.",
  "Enable two-factor authentication in Security settings.",
  "Business hours are 9am to 5pm, Monday through Friday.",
]
idx = SemanticIndex(corpus)
for doc, s in idx.search("how do I recover my login?", k=2):
    print(f"{s:>6}  {doc}")

## 4. Dimensionality = a storage/cost dial
Index size scales linearly with dimensions. Matryoshka models let you truncate.

In [ ]:
for dims in [256, 512, 1536, 3072]:
    bytes_per_vec = dims * 4                 # float32
    for n in [100_000, 2_000_000]:
        gb = n * bytes_per_vec / 1e9
        print(f"dims={dims:4d}  N={n:>9,}  index≈{gb:6.2f} GB (float32)")

## 5. Dense misses exact terms — motivate hybrid search
Dense embeddings match meaning, not exact identifiers. Sparse/BM25 covers that gap.

In [ ]:
docs = ["Error code TS-999 indicates a timeout in the sync service.",
        "The system experienced a general connection problem last night."]
D = embed(docs); q = embed(["TS-999"])[0]
for doc, s in sorted(zip(docs, D @ q), key=lambda x:-x[1]):
    print(f"{s:>6}  {doc}")
print("\nExact IDs like 'TS-999' are where you add BM25/sparse (hybrid search).")

## 6. Exercises
1. With an API key, rank 5 sentences of your own by similarity to a query.
2. Truncate `text-embedding-3-small` to 256 dims via `dims=`; does ranking change much?
3. Compute index size for 5M chunks at 1024 dims; then at int8 quantization (1 byte/dim).
4. Build a tiny hybrid ranker: average normalized dense score with a keyword-overlap score.
5. Embed a sentence and its translation with a multilingual model — is similarity high?

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Embeddings`
- Snippets: `04 Code Snippets/LLM/Generating and Comparing Embeddings`, `.../Semantic Search Over a Corpus`
- MOC: `06 Maps of Content/LLM Engineering Concepts`